# 04a — Why Linear Fails

## The One-Sentence Version
When the structure in your data is curved, straight-line methods can't find it.

## What You'll Build Intuition For
- Why PCA fails on curved manifolds (Swiss roll, circles, S-curve)
- The difference between linear projection and manifold unfolding
- Why we need nonlinear methods

## Prerequisites
- [03a — Projection Intuition](../03_linear_methods/03a_projection_intuition.ipynb)
- [03c — PCA Applied](../03_linear_methods/03c_pca_applied.ipynb) — PCA's Swiss roll failure

## The Story

PCA is a rotation. It finds the best straight lines through your data cloud. When the data varies along straight lines, this is perfect.

But real data is often curved. A Swiss roll is a flat sheet rolled up in 3D space. A PCA projection crushes it flat instead of unrolling it — mixing together points that are far apart on the actual surface. Two concentric circles have a clean nonlinear separation that PCA projects into a useless overlap.

This notebook is a catalogue of PCA's failures. Not to criticise PCA — it's an excellent tool for linear structure — but to motivate the nonlinear methods we'll learn next.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.datasets import make_s_curve

from utils.plotting import apply_style, COLOURS
from utils.data_generators import make_swiss_roll_data

apply_style()
rng = np.random.default_rng(42)

## The Swiss Roll Problem

The Swiss roll is a 2D surface curled up in 3D. It's intrinsically 2D — two numbers (position along the roll, height) describe every point. PCA should reduce it to 2D. Let's see what happens.

In [ ]:
# Swiss roll in 3D
X_swiss, colour_swiss = make_swiss_roll_data(n=2000, noise=0.3, seed=42)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=X_swiss[:, 0], y=X_swiss[:, 1], z=X_swiss[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=colour_swiss, colorscale="Viridis", opacity=0.7),
)])
fig_3d.update_layout(
    title="Swiss Roll: a flat 2D sheet rolled up in 3D",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=700, height=500, margin=dict(l=0, r=0, t=40, b=0),
)
fig_3d.show()

print("Colour encodes position along the roll.")
print("Nearby colours = nearby on the manifold surface.")

In [ ]:
# PCA projection: crushes the roll flat
pca_swiss = PCA(n_components=2)
X_swiss_pca = pca_swiss.fit_transform(X_swiss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], s=3, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")
ax1.set_title("PCA: crushes the roll flat")

# What we want: unrolled coordinates
ax2.scatter(colour_swiss, X_swiss[:, 1], s=3, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax2.set_xlabel("Position along roll")
ax2.set_ylabel("Height")
ax2.set_title("Ideal: unrolled manifold")

fig.suptitle("PCA can't unroll — it projects through the layers", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04a_swiss_roll_pca.png", dpi=150, bbox_inches="tight")
plt.show()

print("Left: PCA mixes colours — distant points on the roll overlap.")
print("Right: the true 2D coordinates keep nearby colours together.")

## Concentric Circles

Two classes arranged in concentric circles. A human instantly sees the pattern. PCA can't — it projects both circles onto the same line, destroying the separation.

In [ ]:
# Generate concentric circles
n_per_class = 300

# Inner circle
theta_inner = rng.uniform(0, 2 * np.pi, n_per_class)
r_inner = 1.0 + rng.normal(0, 0.1, n_per_class)
x_inner = r_inner * np.cos(theta_inner)
y_inner = r_inner * np.sin(theta_inner)

# Outer circle
theta_outer = rng.uniform(0, 2 * np.pi, n_per_class)
r_outer = 3.0 + rng.normal(0, 0.1, n_per_class)
x_outer = r_outer * np.cos(theta_outer)
y_outer = r_outer * np.sin(theta_outer)

X_circles = np.vstack([
    np.column_stack([x_inner, y_inner]),
    np.column_stack([x_outer, y_outer]),
])
y_circles = np.array([0]*n_per_class + [1]*n_per_class)

# PCA projection to 1D
pca_circles = PCA(n_components=1)
X_circles_1d = pca_circles.fit_transform(X_circles)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Original 2D
for cls, colour, label in [(0, COLOURS["signal"], "Inner"),
                            (1, COLOURS["noise"], "Outer")]:
    mask = y_circles == cls
    ax1.scatter(X_circles[mask, 0], X_circles[mask, 1], s=10, alpha=0.5,
               color=colour, label=label)
ax1.set_xlabel("X")
ax1.set_ylabel("Y")
ax1.set_title("Original: clearly separable")
ax1.legend()
ax1.set_aspect("equal")

# PCA 1D projection
for cls, colour, label in [(0, COLOURS["signal"], "Inner"),
                            (1, COLOURS["noise"], "Outer")]:
    mask = y_circles == cls
    ax2.hist(X_circles_1d[mask].ravel(), bins=30, alpha=0.5,
             color=colour, label=label, edgecolor="white", linewidth=0.3)
ax2.set_xlabel("PC1")
ax2.set_title("PCA 1D: classes overlap completely")
ax2.legend()

fig.suptitle("PCA can't find nonlinear boundaries", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04a_concentric_circles.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA projects both circles onto the same axis — no separation.")
print("The structure is radial (nonlinear), not directional (linear).")

## The S-Curve

Another classic nonlinear manifold: an S-shaped surface in 3D. Like the Swiss roll, it's intrinsically 2D but PCA can't unfold it.

In [ ]:
# S-curve: 2D manifold in 3D
X_scurve, colour_scurve = make_s_curve(n_samples=1500, noise=0.1, random_state=42)

# PCA projection
pca_scurve = PCA(n_components=2)
X_scurve_pca = pca_scurve.fit_transform(X_scurve)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# 3D view (static — two axes)
ax1.scatter(X_scurve[:, 0], X_scurve[:, 2], s=5, alpha=0.5,
            c=colour_scurve, cmap="viridis")
ax1.set_xlabel("X")
ax1.set_ylabel("Z")
ax1.set_title("S-Curve (X-Z view)")

# PCA projection
ax2.scatter(X_scurve_pca[:, 0], X_scurve_pca[:, 1], s=5, alpha=0.5,
            c=colour_scurve, cmap="viridis")
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title("PCA: folds onto itself")

# Ideal unfolding
ax3.scatter(colour_scurve, X_scurve[:, 1], s=5, alpha=0.5,
            c=colour_scurve, cmap="viridis")
ax3.set_xlabel("Position along S")
ax3.set_ylabel("Height")
ax3.set_title("Ideal: unfolded S-curve")

fig.suptitle("PCA folds the S-curve — it can't follow the curvature", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04a_s_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## What Goes Wrong Formally

PCA finds **linear combinations** of the original features. Every principal component is a weighted sum:

$$z_k = w_{k1} x_1 + w_{k2} x_2 + \ldots + w_{kd} x_d$$

If the true low-dimensional coordinates are a **nonlinear** function of the features — say, the angle around a spiral, or the distance from a centre — no linear combination can capture them.

**Analogy:** Imagine trying to unroll a piece of paper using only rigid rotations and reflections. You can tilt it, flip it, but you can never flatten a rolled-up sheet by rotating it. You need to *bend* it back — and that's a nonlinear operation.

In [ ]:
# Quantify the failure: neighbourhood preservation
# For each point, compare its k nearest neighbours in the original space
# vs in the PCA projection
from sklearn.neighbors import NearestNeighbors

def neighbourhood_preservation(X_original, X_projected, k=10):
    """Fraction of k-nearest neighbours preserved after projection."""
    nn_orig = NearestNeighbors(n_neighbors=k+1).fit(X_original)
    nn_proj = NearestNeighbors(n_neighbors=k+1).fit(X_projected)

    _, idx_orig = nn_orig.kneighbors(X_original)
    _, idx_proj = nn_proj.kneighbors(X_projected)

    preserved = 0
    for i in range(len(X_original)):
        orig_set = set(idx_orig[i, 1:])   # exclude self
        proj_set = set(idx_proj[i, 1:])
        preserved += len(orig_set & proj_set) / k

    return preserved / len(X_original)

# Measure on Swiss roll and S-curve
swiss_preservation = neighbourhood_preservation(X_swiss, X_swiss_pca, k=10)
scurve_preservation = neighbourhood_preservation(X_scurve, X_scurve_pca, k=10)

# Baseline: linear data (PCA should be perfect)
X_linear = rng.normal(0, 1, (1500, 3)) @ np.diag([3, 1, 0.1])
X_linear_pca = PCA(n_components=2).fit_transform(X_linear)
linear_preservation = neighbourhood_preservation(X_linear, X_linear_pca, k=10)

datasets = ["Linear cloud", "Swiss roll", "S-curve"]
scores = [linear_preservation, swiss_preservation, scurve_preservation]
bar_colours = [COLOURS["success"], COLOURS["noise"], COLOURS["noise"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(datasets, scores, color=bar_colours, alpha=0.8, edgecolor="white")
ax.set_xlabel("Neighbourhood Preservation (10-NN)")
ax.set_title("PCA preserves neighbourhoods on linear data, fails on curved data")
ax.set_xlim(0, 1)

for i, s in enumerate(scores):
    ax.text(s + 0.02, i, f"{s:.0%}", va="center", fontsize=11)

plt.tight_layout()
plt.savefig("visuals/04a_neighbourhood_preservation.png", dpi=150, bbox_inches="tight")
plt.show()

> **Key insight:** PCA's failure on curved data isn't a bug — it's a fundamental limitation. Linear methods can only find structure that lies along straight lines. When the manifold is curved, you need methods that can measure distance *along the surface*, not through the air. That's what t-SNE, UMAP, and Isomap do.

<details>
<summary><b>The Maths Behind This</b></summary>

PCA minimises: $\min_W \sum_i \|\mathbf{x}_i - WW^T\mathbf{x}_i\|^2$ where $W$ has orthonormal columns.

This is equivalent to finding the best $k$-dimensional **affine subspace**. For the Swiss roll, the data lies on a 2D manifold that is *not* an affine subspace — it's a curved surface. The best-fitting plane passes through the roll, projecting points from different layers onto each other.

Nonlinear methods replace Euclidean distance $\|\mathbf{x}_i - \mathbf{x}_j\|$ with **geodesic distance** (shortest path along the manifold). Isomap builds a graph of nearest neighbours and computes shortest paths. t-SNE and UMAP use local neighbourhood structure to build a similarity graph that respects the manifold topology.

</details>

## Where This Matters

**Single-cell genomics:** Gene expression data from individual cells often has complex, branching manifold structure (cell differentiation pathways). PCA gives a blurry view; nonlinear methods reveal the branches.

**Sensor networks:** A robot's sensor readings as it moves through a room create a manifold parameterised by position and orientation. PCA can't untangle the room's geometry from the sensor values.

## Feynman Check

1. **Why can't PCA unroll a Swiss roll?**

2. **Give an example of data where PCA would work perfectly and one where it would fail.**

Now that you understand *why* we need nonlinear methods, let's learn the most popular one: **04b — t-SNE Explained**.